# MosquitoWatch AI — Kaggle Training Notebook
### Layer 1: Drone-based AI vision model with YOLO26s

Pipeline:

**Dataset → Data quality → Leakage check → Class balance → Targeted augmentation → YOLO26s training → Underfitting/overfitting checks → Internal test → Per-class evaluation → Small-object evaluation → Confidence-threshold tuning → Robustness test → External generalization test → Speed test → final `best.pt`**

Important:
- Add your **main YOLO-format drone dataset** to Kaggle Inputs.
- Optionally add a **second YOLO-format drone dataset** for external generalization testing.
- This notebook trains **Layer 1 only**. Layer 2 will later use the final object counts + weather data.
- Validation/test data are never augmented.


In [ ]:
# CELL 1 — Install required packages
!pip install -q -U ultralytics albumentations pyyaml


In [ ]:
# CELL 2 — Imports, versions, GPU check, and random seed

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import os
import random
import shutil
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from IPython.display import display

from ultralytics import YOLO
import ultralytics
import albumentations as A

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Enable a Kaggle GPU before training.")


In [ ]:
# CELL 3 — Find all YOLO data.yaml files attached to Kaggle

INPUT_ROOT = Path("/kaggle/input")

yaml_candidates = sorted(
    list(INPUT_ROOT.rglob("data.yaml")) +
    list(INPUT_ROOT.rglob("dataset.yaml"))
)

yaml_candidates = list(dict.fromkeys(yaml_candidates))

print(f"Found {len(yaml_candidates)} YOLO dataset YAML file(s):")

for i, path in enumerate(yaml_candidates):
    print(f"[{i}] {path}")

if not yaml_candidates:
    raise FileNotFoundError(
        "No data.yaml/dataset.yaml found. Add a YOLO-format dataset to Kaggle Inputs."
    )


In [ ]:
# CELL 4 — Choose the main dataset and optional external dataset

# After running CELL 3, change only these indices if needed.
MAIN_YAML_INDEX = 0

# Use the second attached dataset for generalization if available.
# Set to None if you only attached one dataset.
EXTERNAL_YAML_INDEX = 1 if len(yaml_candidates) > 1 else None

MAIN_YAML = yaml_candidates[MAIN_YAML_INDEX]

EXTERNAL_YAML = (
    yaml_candidates[EXTERNAL_YAML_INDEX]
    if EXTERNAL_YAML_INDEX is not None
    else None
)

print("Main dataset:", MAIN_YAML)
print("External dataset:", EXTERNAL_YAML)


In [ ]:
# CELL 5 — Dataset helper functions

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def normalize_names(names):
    if isinstance(names, list):
        return {i: str(name) for i, name in enumerate(names)}

    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}

    raise ValueError("Unsupported 'names' format in YAML.")

def load_dataset_yaml(yaml_path):
    yaml_path = Path(yaml_path)

    with open(yaml_path, "r") as f:
        cfg = yaml.safe_load(f)

    names = normalize_names(cfg["names"])

    root = Path(cfg.get("path", ""))

    if not root.is_absolute():
        root = (yaml_path.parent / root).resolve()

    def resolve_split(split_name):
        value = cfg.get(split_name)

        if value is None:
            return None

        if isinstance(value, list):
            raise ValueError(
                f"{split_name} uses multiple paths. "
                "This notebook currently expects one image directory per split."
            )

        split_path = Path(value)

        if not split_path.is_absolute():
            split_path = (root / split_path).resolve()

        return split_path

    return {
        "yaml": yaml_path,
        "root": root,
        "names": names,
        "train_images": resolve_split("train"),
        "val_images": resolve_split("val"),
        "test_images": resolve_split("test"),
        "raw": cfg,
    }

def image_dir_to_label_dir(image_dir):
    image_dir = Path(image_dir)

    if image_dir.name.lower() == "images":
        return image_dir.parent / "labels"

    parts = list(image_dir.parts)

    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")
        parts[idx] = "labels"
        return Path(*parts)

    raise ValueError(f"Could not infer labels directory from: {image_dir}")

def list_images(image_dir):
    if image_dir is None or not Path(image_dir).exists():
        return []

    return sorted(
        p for p in Path(image_dir).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

main_ds = load_dataset_yaml(MAIN_YAML)

print("Classes:")
for cid, name in main_ds["names"].items():
    print(f"  {cid}: {name}")

print("\nSplits:")
print("Train:", main_ds["train_images"])
print("Val:  ", main_ds["val_images"])
print("Test: ", main_ds["test_images"])


In [ ]:
# CELL 6 — Scan image and annotation quality

def read_yolo_labels(label_path):
    rows = []

    if not label_path.exists():
        return rows

    text = label_path.read_text().strip()

    if not text:
        return rows

    for line_no, line in enumerate(text.splitlines(), start=1):
        parts = line.split()

        if len(parts) != 5:
            raise ValueError(
                f"{label_path} line {line_no}: expected 5 values, got {len(parts)}"
            )

        class_id = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:])

        rows.append((class_id, x, y, w, h))

    return rows

def scan_split(image_dir, class_names, split_name):
    image_dir = Path(image_dir)
    label_dir = image_dir_to_label_dir(image_dir)
    images = list_images(image_dir)

    report = {
        "split": split_name,
        "images": len(images),
        "readable_images": 0,
        "corrupted_images": [],
        "missing_labels": [],
        "empty_labels": [],
        "invalid_labels": [],
        "class_counts": Counter(),
        "size_counts": Counter(),
        "resolutions": Counter(),
    }

    for img_path in images:

        # Image readability
        try:
            with Image.open(img_path) as im:
                im.verify()

            with Image.open(img_path) as im:
                width, height = im.size

            report["readable_images"] += 1
            report["resolutions"][(width, height)] += 1

        except Exception as e:
            report["corrupted_images"].append((str(img_path), str(e)))
            continue

        # Matching YOLO label file
        rel = img_path.relative_to(image_dir)
        label_path = (label_dir / rel).with_suffix(".txt")

        if not label_path.exists():
            report["missing_labels"].append(str(img_path))
            continue

        if label_path.read_text().strip() == "":
            report["empty_labels"].append(str(label_path))
            continue

        try:
            labels = read_yolo_labels(label_path)

        except Exception as e:
            report["invalid_labels"].append((str(label_path), str(e)))
            continue

        for class_id, x, y, w, h in labels:

            if class_id not in class_names:
                report["invalid_labels"].append(
                    (str(label_path), f"invalid class id {class_id}")
                )
                continue

            valid = (
                0 <= x <= 1 and
                0 <= y <= 1 and
                0 < w <= 1 and
                0 < h <= 1
            )

            if not valid:
                report["invalid_labels"].append(
                    (str(label_path), f"invalid box: {class_id} {x} {y} {w} {h}")
                )
                continue

            report["class_counts"][class_id] += 1

            # Approximate COCO-style size groups in original pixels
            box_w_px = w * width
            box_h_px = h * height
            area_px = box_w_px * box_h_px

            if area_px < 32**2:
                size_name = "small"
            elif area_px < 96**2:
                size_name = "medium"
            else:
                size_name = "large"

            report["size_counts"][size_name] += 1

    return report

quality_reports = {}

for split_name, split_path in [
    ("train", main_ds["train_images"]),
    ("val", main_ds["val_images"]),
    ("test", main_ds["test_images"]),
]:

    if split_path is not None and Path(split_path).exists():
        quality_reports[split_name] = scan_split(
            split_path,
            main_ds["names"],
            split_name
        )

summary_rows = []

for split_name, r in quality_reports.items():
    summary_rows.append({
        "split": split_name,
        "images": r["images"],
        "readable": r["readable_images"],
        "corrupted": len(r["corrupted_images"]),
        "missing_labels": len(r["missing_labels"]),
        "empty_labels": len(r["empty_labels"]),
        "invalid_label_entries": len(r["invalid_labels"]),
        "small_objects": r["size_counts"]["small"],
        "medium_objects": r["size_counts"]["medium"],
        "large_objects": r["size_counts"]["large"],
    })

quality_df = pd.DataFrame(summary_rows)
display(quality_df)

print("\nInterpretation:")
print("- Corrupted images should be removed/replaced.")
print("- Invalid labels should be fixed before training.")
print("- Missing labels can be valid background images, but inspect them.")


In [ ]:
# CELL 7 — Show class distribution before augmentation

train_counts = quality_reports["train"]["class_counts"]

class_rows = []

for cid, name in main_ds["names"].items():
    class_rows.append({
        "class_id": cid,
        "class_name": name,
        "train_boxes": train_counts.get(cid, 0),
    })

class_df_before = pd.DataFrame(class_rows)
display(class_df_before)

plt.figure(figsize=(10, 5))
plt.bar(class_df_before["class_name"], class_df_before["train_boxes"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Number of bounding boxes")
plt.title("Training class distribution — before targeted augmentation")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 8 — Check exact duplicate images across train/validation/test

def file_md5(path, chunk_size=1024 * 1024):
    md5 = hashlib.md5()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()

hash_locations = defaultdict(list)

for split_name, split_path in [
    ("train", main_ds["train_images"]),
    ("val", main_ds["val_images"]),
    ("test", main_ds["test_images"]),
]:

    if split_path is None:
        continue

    for img_path in list_images(split_path):
        hash_locations[file_md5(img_path)].append(
            (split_name, str(img_path))
        )

cross_split_duplicates = []

for img_hash, locations in hash_locations.items():

    splits = {x[0] for x in locations}

    if len(splits) > 1:
        cross_split_duplicates.append(
            (img_hash, locations)
        )

print("Exact duplicate groups crossing splits:",
      len(cross_split_duplicates))

for item in cross_split_duplicates[:10]:

    print("\nDuplicate group:")

    for loc in item[1]:
        print(" ", loc)

if cross_split_duplicates:
    print("\nWARNING: Fix these before trusting test metrics.")
else:
    print("\nGood: no exact duplicate images found across splits.")


## CELL 9 — Split-by-flight/site/video rule

If the dataset provides **flight ID, site ID, or source-video ID**, keep one group in only one split.

Good:
- Flight A → Train
- Flight B → Validation
- Flight C → Test

Bad:
- Frame 101 → Train
- Frame 102 → Validation
- Frame 103 → Test

Adjacent drone-video frames can look almost identical and cause data leakage.


In [ ]:
# CELL 9 — Show current split sizes

split_sizes = {
    "train": len(list_images(main_ds["train_images"]))
             if main_ds["train_images"] else 0,

    "val": len(list_images(main_ds["val_images"]))
           if main_ds["val_images"] else 0,

    "test": len(list_images(main_ds["test_images"]))
            if main_ds["test_images"] else 0,
}

for split_name, n in split_sizes.items():
    print(f"{split_name:>5}: {n} images")


In [ ]:
# CELL 10 — Create writable working dataset using symbolic links

WORK_ROOT = Path("/kaggle/working/mosquitowatch_dataset")

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

for split_name, source_images in [
    ("train", main_ds["train_images"]),
    ("val", main_ds["val_images"]),
    ("test", main_ds["test_images"]),
]:

    if source_images is None or not Path(source_images).exists():
        continue

    source_images = Path(source_images)
    source_labels = image_dir_to_label_dir(source_images)

    target_images = WORK_ROOT / split_name / "images"
    target_labels = WORK_ROOT / split_name / "labels"

    target_images.mkdir(parents=True, exist_ok=True)
    target_labels.mkdir(parents=True, exist_ok=True)

    for img_path in list_images(source_images):

        rel = img_path.relative_to(source_images)
        src_label = (source_labels / rel).with_suffix(".txt")

        # Flatten subfolders so filenames remain unique in working folder.
        flat_name = "__".join(rel.parts)

        dst_img = target_images / flat_name
        dst_label = target_labels / Path(flat_name).with_suffix(".txt")

        os.symlink(img_path, dst_img)

        if src_label.exists():
            os.symlink(src_label, dst_label)
        else:
            dst_label.write_text("")

print("Working dataset:", WORK_ROOT)
print("Train images:", len(list_images(WORK_ROOT / "train/images")))
print("Val images:  ", len(list_images(WORK_ROOT / "val/images")))
print("Test images: ",
      len(list_images(WORK_ROOT / "test/images"))
      if (WORK_ROOT / "test/images").exists() else 0)


In [ ]:
# CELL 11 — Define targeted augmentation for minority classes

TARGET_RATIO_OF_MEDIAN = 1.0
MAX_AUGMENTED_IMAGES = len(
    list_images(WORK_ROOT / "train/images")
)
MAX_ATTEMPTS_PER_IMAGE = 4

augmenter = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        A.RandomBrightnessContrast(
            brightness_limit=0.20,
            contrast_limit=0.20,
            p=0.5,
        ),

        A.Affine(
            scale=(0.90, 1.10),
            translate_percent=(-0.05, 0.05),
            rotate=(-15, 15),
            shear=(-5, 5),
            p=0.7,
        ),

        A.GaussianBlur(
            blur_limit=(3, 5),
            p=0.12
        ),
    ],

    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.30,
        clip=True,
    ),
)

def load_image_and_labels(img_path, label_path):
    image = cv2.imread(str(img_path))

    if image is None:
        raise ValueError(f"Could not read: {img_path}")

    labels = read_yolo_labels(label_path)

    bboxes = [
        [x, y, w, h]
        for cid, x, y, w, h in labels
    ]

    class_labels = [
        cid
        for cid, x, y, w, h in labels
    ]

    return image, bboxes, class_labels

def save_yolo_labels(label_path, bboxes, class_labels):
    lines = []

    for cid, (x, y, w, h) in zip(class_labels, bboxes):
        lines.append(
            f"{int(cid)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
        )

    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else "")
    )

train_img_dir = WORK_ROOT / "train/images"
train_lbl_dir = WORK_ROOT / "train/labels"

train_image_info = []
working_counts_before = Counter()

for img_path in list_images(train_img_dir):

    label_path = (
        train_lbl_dir /
        img_path.with_suffix(".txt").name
    )

    labels = read_yolo_labels(label_path)

    classes = [
        row[0]
        for row in labels
    ]

    for cid in classes:
        working_counts_before[cid] += 1

    train_image_info.append({
        "image": img_path,
        "label": label_path,
        "classes": set(classes),
    })

nonzero_counts = [
    c for c in working_counts_before.values()
    if c > 0
]

target_count = int(
    np.median(nonzero_counts) *
    TARGET_RATIO_OF_MEDIAN
)

minority_classes = {
    cid
    for cid in main_ds["names"]
    if working_counts_before.get(cid, 0) < target_count
}

print("Target minimum count:", target_count)

print("\nMinority classes:")
for cid in sorted(minority_classes):
    print(
        f"{cid}: {main_ds['names'][cid]} "
        f"({working_counts_before.get(cid, 0)} boxes)"
    )


In [ ]:
# CELL 12 — Run targeted offline augmentation

current_counts = working_counts_before.copy()

generated = 0
attempts = 0

images_by_class = defaultdict(list)

for info in train_image_info:
    for cid in info["classes"]:
        images_by_class[cid].append(info)

def class_deficit(cid):
    return max(
        0,
        target_count -
        current_counts.get(cid, 0)
    )

while generated < MAX_AUGMENTED_IMAGES:

    deficits = {
        cid: class_deficit(cid)
        for cid in minority_classes
    }

    active = [
        cid
        for cid, deficit in deficits.items()
        if deficit > 0
    ]

    if not active:
        break

    # Class with the largest remaining shortage
    target_cid = max(
        active,
        key=lambda cid: deficits[cid]
    )

    candidates = images_by_class.get(
        target_cid,
        []
    )

    if not candidates:
        minority_classes.remove(target_cid)
        continue

    source = random.choice(candidates)

    success = False

    for _ in range(MAX_ATTEMPTS_PER_IMAGE):

        image, bboxes, class_labels = (
            load_image_and_labels(
                source["image"],
                source["label"]
            )
        )

        transformed = augmenter(
            image=image,
            bboxes=bboxes,
            class_labels=class_labels,
        )

        new_boxes = transformed["bboxes"]
        new_classes = transformed["class_labels"]

        if (
            target_cid in new_classes
            and len(new_boxes) > 0
        ):
            success = True
            break

    attempts += 1

    if not success:

        if attempts > MAX_AUGMENTED_IMAGES * 10:
            print("Stopping: too many failed augmentation attempts.")
            break

        continue

    stem = source["image"].stem

    aug_name = (
        f"aug_{generated:05d}_{stem}.jpg"
    )

    aug_img_path = (
        train_img_dir /
        aug_name
    )

    aug_lbl_path = (
        train_lbl_dir /
        f"aug_{generated:05d}_{stem}.txt"
    )

    cv2.imwrite(
        str(aug_img_path),
        transformed["image"]
    )

    save_yolo_labels(
        aug_lbl_path,
        new_boxes,
        new_classes
    )

    for cid in new_classes:
        current_counts[int(cid)] += 1

    generated += 1

print("Augmented images created:", generated)

print(
    "Training images BEFORE:",
    len(train_image_info)
)

print(
    "Training images AFTER:",
    len(list_images(train_img_dir))
)


In [ ]:
# CELL 13 — Before vs after augmentation report

before_after_rows = []

for cid, name in main_ds["names"].items():

    before_after_rows.append({
        "class_id": cid,
        "class_name": name,
        "before_boxes": working_counts_before.get(cid, 0),
        "after_boxes": current_counts.get(cid, 0),
        "change": (
            current_counts.get(cid, 0) -
            working_counts_before.get(cid, 0)
        ),
    })

augmentation_df = pd.DataFrame(
    before_after_rows
)

display(augmentation_df)

print(
    f"Training images: "
    f"{len(train_image_info)} → "
    f"{len(list_images(train_img_dir))}"
)

x = np.arange(len(augmentation_df))
width = 0.38

plt.figure(figsize=(11, 5))

plt.bar(
    x - width/2,
    augmentation_df["before_boxes"],
    width,
    label="Before"
)

plt.bar(
    x + width/2,
    augmentation_df["after_boxes"],
    width,
    label="After"
)

plt.xticks(
    x,
    augmentation_df["class_name"],
    rotation=35,
    ha="right"
)

plt.ylabel("Bounding boxes")
plt.title(
    "Class distribution before vs after targeted augmentation"
)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# CELL 14 — Write final augmented data.yaml

AUG_YAML = WORK_ROOT / "data_aug.yaml"

yaml_data = {
    "path": str(WORK_ROOT),
    "train": "train/images",
    "val": "val/images",
    "names": main_ds["names"],
}

if (WORK_ROOT / "test/images").exists():
    yaml_data["test"] = "test/images"

with open(AUG_YAML, "w") as f:
    yaml.safe_dump(
        yaml_data,
        f,
        sort_keys=False
    )

print(AUG_YAML.read_text())


## CELL 15 — Training settings

Start with:
- **YOLO26s**
- **1024 px** input size for small aerial objects
- **100 epochs**
- early stopping with patience
- moderate online augmentation

If Kaggle runs out of GPU memory:
1. reduce `BATCH_SIZE` from 8 to 4;
2. if needed, reduce `IMAGE_SIZE` from 1024 to 768.


In [ ]:
# CELL 15 — Train YOLO26s

MODEL_WEIGHTS = "yolo26s.pt"

IMAGE_SIZE = 1024
BATCH_SIZE = 8
EPOCHS = 100
PATIENCE = 20

RUNS_DIR = Path(
    "/kaggle/working/runs"
)

RUN_NAME = (
    "mosquitowatch_yolo26s"
)

model = YOLO(
    MODEL_WEIGHTS
)

train_results = model.train(
    data=str(AUG_YAML),

    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,

    patience=PATIENCE,

    device=(
        0
        if torch.cuda.is_available()
        else "cpu"
    ),

    workers=2,
    seed=SEED,
    deterministic=True,

    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,

    plots=True,

    # Moderate online augmentation.
    # Offline augmentation already targeted minority classes.
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.10,

    degrees=10.0,
    translate=0.10,
    scale=0.50,

    fliplr=0.50,
    flipud=0.50,

    close_mosaic=10,
)

RUN_DIR = Path(
    model.trainer.save_dir
)

BEST_PT = (
    RUN_DIR /
    "weights" /
    "best.pt"
)

LAST_PT = (
    RUN_DIR /
    "weights" /
    "last.pt"
)

print("Run directory:", RUN_DIR)
print("Best model:", BEST_PT)
print("Best model exists:", BEST_PT.exists())


In [ ]:
# CELL 16 — Plot training and validation curves

RESULTS_CSV = (
    RUN_DIR /
    "results.csv"
)

history = pd.read_csv(
    RESULTS_CSV
)

history.columns = [
    c.strip()
    for c in history.columns
]

display(
    history.tail()
)

train_loss_cols = [
    c
    for c in history.columns
    if c.startswith("train/")
    and "loss" in c
]

val_loss_cols = [
    c
    for c in history.columns
    if c.startswith("val/")
    and "loss" in c
]

history["train_total_loss"] = (
    history[train_loss_cols]
    .sum(axis=1)
)

history["val_total_loss"] = (
    history[val_loss_cols]
    .sum(axis=1)
)

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["train_total_loss"],
    label="Train total loss"
)

plt.plot(
    history["epoch"],
    history["val_total_loss"],
    label="Validation total loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs validation loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

metric_col = next(
    (
        c
        for c in history.columns
        if "mAP50-95" in c
    ),
    None
)

if metric_col:

    plt.figure(figsize=(10, 5))

    plt.plot(
        history["epoch"],
        history[metric_col]
    )

    plt.xlabel("Epoch")
    plt.ylabel("mAP50-95")
    plt.title(
        "Validation mAP50-95 by epoch"
    )
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:
# CELL 17 — Evaluate best.pt on train, validation, and test

best_model = YOLO(
    str(BEST_PT)
)

def evaluate_split(
    model,
    data_yaml,
    split,
    conf=0.001
):

    metrics = model.val(
        data=str(data_yaml),
        split=split,

        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,

        conf=conf,

        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),

        plots=True,
        verbose=False,

        project=str(RUNS_DIR),
        name=f"eval_{split}",
        exist_ok=True,
    )

    return {
        "split": split,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
    }, metrics

eval_rows = []
eval_objects = {}

for split in [
    "train",
    "val",
    "test"
]:

    split_path = yaml_data.get(split)

    if split_path is None:
        continue

    row, obj = evaluate_split(
        best_model,
        AUG_YAML,
        split
    )

    eval_rows.append(row)
    eval_objects[split] = obj

fit_metrics_df = pd.DataFrame(
    eval_rows
)

display(
    fit_metrics_df
)


In [ ]:
# CELL 18 — Underfitting and overfitting diagnostic

# Diagnostic warning thresholds.
# These are NOT universal scientific cutoffs.
UNDERFIT_MAP50_WARNING = 0.60
OVERFIT_GAP_WARNING = 0.15

metrics_by_split = (
    fit_metrics_df
    .set_index("split")
)

train_map = (
    metrics_by_split
    .loc["train", "mAP50"]
)

val_map = (
    metrics_by_split
    .loc["val", "mAP50"]
)

test_map = (
    metrics_by_split
    .loc["test", "mAP50"]
    if "test" in metrics_by_split.index
    else np.nan
)

print(
    f"Train mAP50: {train_map:.3f}"
)

print(
    f"Val   mAP50: {val_map:.3f}"
)

if not np.isnan(test_map):
    print(
        f"Test  mAP50: {test_map:.3f}"
    )

underfit_flag = (
    train_map < UNDERFIT_MAP50_WARNING
    and
    val_map < UNDERFIT_MAP50_WARNING
)

overfit_gap_val = (
    train_map - val_map
)

overfit_gap_test = (
    train_map - test_map
    if not np.isnan(test_map)
    else 0
)

overfit_flag = (
    overfit_gap_val > OVERFIT_GAP_WARNING
    or
    overfit_gap_test > OVERFIT_GAP_WARNING
)

print("\nFIT DIAGNOSIS")

if underfit_flag:

    print(
        "⚠ POSSIBLE UNDERFITTING"
    )

    print(
        "Training and validation performance are both low."
    )

elif overfit_flag:

    print(
        "⚠ POSSIBLE OVERFITTING"
    )

    print(
        "Training performance is much higher than unseen data."
    )

else:

    print(
        "✓ No strong underfitting/overfitting warning."
    )

print(
    "\nTrain − Val mAP50 gap:",
    round(overfit_gap_val, 3)
)

if not np.isnan(test_map):

    print(
        "Train − Test mAP50 gap:",
        round(overfit_gap_test, 3)
    )

# Additional late-epoch overfitting signal
map_col = next(
    (
        c
        for c in history.columns
        if "mAP50-95" in c
    ),
    None
)

if map_col:

    best_epoch_idx = (
        history[map_col]
        .idxmax()
    )

    best_epoch = int(
        history.loc[
            best_epoch_idx,
            "epoch"
        ]
    )

    peak_map = float(
        history.loc[
            best_epoch_idx,
            map_col
        ]
    )

    final_map = float(
        history.iloc[-1][map_col]
    )

    print(
        "\nBest validation epoch:",
        best_epoch
    )

    print(
        "Peak val mAP50-95:",
        round(peak_map, 3)
    )

    print(
        "Final val mAP50-95:",
        round(final_map, 3)
    )

    if peak_map - final_map > 0.03:
        print(
            "⚠ Validation performance dropped after its peak."
        )


In [ ]:
# CELL 19 — Per-class evaluation on internal test set

if "test" not in eval_objects:
    raise RuntimeError(
        "A labeled test split is required."
    )

test_metrics = (
    eval_objects["test"]
)

p_values = np.asarray(
    test_metrics.box.p
)

r_values = np.asarray(
    test_metrics.box.r
)

map_values = np.asarray(
    test_metrics.box.maps
)

per_class_rows = []

for cid, name in main_ds["names"].items():

    if cid < len(map_values):

        per_class_rows.append({
            "class_id": cid,
            "class_name": name,

            "precision": (
                float(p_values[cid])
                if cid < len(p_values)
                else np.nan
            ),

            "recall": (
                float(r_values[cid])
                if cid < len(r_values)
                else np.nan
            ),

            "mAP50-95": float(
                map_values[cid]
            ),
        })

per_class_df = pd.DataFrame(
    per_class_rows
)

display(
    per_class_df.sort_values(
        "recall"
    )
)

plt.figure(figsize=(10, 5))

plt.bar(
    per_class_df["class_name"],
    per_class_df["recall"]
)

plt.ylim(0, 1)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.ylabel("Recall")
plt.title(
    "Per-class recall — internal test set"
)
plt.tight_layout()
plt.show()


## CELL 20 — Confidence-threshold tuning

Layer 1 will use the selected threshold when counting objects.

Because this is a **government pre-screening system**, we prefer **high recall** so real risk objects are less likely to be missed.


In [ ]:
# CELL 20 — Tune confidence threshold using validation data

CONFIDENCE_CANDIDATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
]

TARGET_RECALL = 0.90

threshold_rows = []

for conf in CONFIDENCE_CANDIDATES:

    metrics = best_model.val(
        data=str(AUG_YAML),
        split="val",

        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,

        conf=conf,

        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),

        plots=False,
        verbose=False,
    )

    precision = float(
        metrics.box.mp
    )

    recall = float(
        metrics.box.mr
    )

    f1 = (
        2 *
        precision *
        recall /
        (
            precision +
            recall +
            1e-12
        )
    )

    threshold_rows.append({
        "confidence": conf,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(
            metrics.box.map50
        ),
        "mAP50-95": float(
            metrics.box.map
        ),
    })

threshold_df = pd.DataFrame(
    threshold_rows
)

display(
    threshold_df
)

eligible = threshold_df[
    threshold_df["recall"] >=
    TARGET_RECALL
]

if len(eligible):

    chosen = (
        eligible
        .sort_values(
            ["precision", "confidence"],
            ascending=[False, False]
        )
        .iloc[0]
    )

else:

    chosen = (
        threshold_df
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

SELECTED_CONF = float(
    chosen["confidence"]
)

print(
    f"Selected confidence threshold: "
    f"{SELECTED_CONF:.2f}"
)

print(
    f"Validation precision: "
    f"{chosen['precision']:.3f}"
)

print(
    f"Validation recall: "
    f"{chosen['recall']:.3f}"
)

print(
    f"Validation F1: "
    f"{chosen['f1']:.3f}"
)


In [ ]:
# CELL 21 — Evaluate recall by object size

def yolo_box_to_xyxy(
    x,
    y,
    w,
    h,
    img_w,
    img_h
):
    x1 = (x - w / 2) * img_w
    y1 = (y - h / 2) * img_h
    x2 = (x + w / 2) * img_w
    y2 = (y + h / 2) * img_h

    return np.array(
        [x1, y1, x2, y2],
        dtype=float
    )

def box_iou(box_a, box_b):

    x1 = max(
        box_a[0],
        box_b[0]
    )

    y1 = max(
        box_a[1],
        box_b[1]
    )

    x2 = min(
        box_a[2],
        box_b[2]
    )

    y2 = min(
        box_a[3],
        box_b[3]
    )

    inter = (
        max(0, x2 - x1) *
        max(0, y2 - y1)
    )

    area_a = (
        max(0, box_a[2] - box_a[0]) *
        max(0, box_a[3] - box_a[1])
    )

    area_b = (
        max(0, box_b[2] - box_b[0]) *
        max(0, box_b[3] - box_b[1])
    )

    union = (
        area_a +
        area_b -
        inter
    )

    return (
        inter / union
        if union > 0
        else 0.0
    )

def size_bucket(pixel_area):

    if pixel_area < 32**2:
        return "small"

    if pixel_area < 96**2:
        return "medium"

    return "large"

test_img_dir = (
    WORK_ROOT /
    "test/images"
)

test_lbl_dir = (
    WORK_ROOT /
    "test/labels"
)

test_images = list_images(
    test_img_dir
)

size_stats = {
    "small": {
        "gt": 0,
        "matched": 0
    },

    "medium": {
        "gt": 0,
        "matched": 0
    },

    "large": {
        "gt": 0,
        "matched": 0
    },
}

for img_path in test_images:

    image = cv2.imread(
        str(img_path)
    )

    h, w = image.shape[:2]

    label_path = (
        test_lbl_dir /
        img_path.with_suffix(".txt").name
    )

    gt_rows = read_yolo_labels(
        label_path
    )

    gt_objects = []

    for cid, xc, yc, bw, bh in gt_rows:

        xyxy = yolo_box_to_xyxy(
            xc,
            yc,
            bw,
            bh,
            w,
            h
        )

        area = (
            (bw * w) *
            (bh * h)
        )

        bucket = size_bucket(
            area
        )

        gt_objects.append({
            "class_id": cid,
            "box": xyxy,
            "size": bucket,
            "matched": False,
        })

        size_stats[bucket]["gt"] += 1

    result = best_model.predict(
        source=image,

        conf=SELECTED_CONF,
        iou=0.60,

        imgsz=IMAGE_SIZE,

        verbose=False,

        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),
    )[0]

    if (
        result.boxes is None
        or
        len(result.boxes) == 0
    ):
        continue

    pred_boxes = (
        result.boxes.xyxy
        .cpu()
        .numpy()
    )

    pred_classes = (
        result.boxes.cls
        .cpu()
        .numpy()
        .astype(int)
    )

    pred_conf = (
        result.boxes.conf
        .cpu()
        .numpy()
    )

    order = np.argsort(
        -pred_conf
    )

    for pi in order:

        pbox = pred_boxes[pi]
        pcid = pred_classes[pi]

        best_match = None
        best_iou = 0.0

        for gi, gt in enumerate(gt_objects):

            if gt["matched"]:
                continue

            if gt["class_id"] != pcid:
                continue

            iou = box_iou(
                pbox,
                gt["box"]
            )

            if (
                iou >= 0.50
                and
                iou > best_iou
            ):
                best_iou = iou
                best_match = gi

        if best_match is not None:

            gt_objects[
                best_match
            ]["matched"] = True

            bucket = gt_objects[
                best_match
            ]["size"]

            size_stats[
                bucket
            ]["matched"] += 1

size_rows = []

for bucket, stats in size_stats.items():

    recall = (
        stats["matched"] /
        stats["gt"]
        if stats["gt"]
        else np.nan
    )

    size_rows.append({
        "object_size": bucket,
        "ground_truth_objects": stats["gt"],
        "detected_at_iou_0.5": stats["matched"],
        "recall": recall,
    })

size_recall_df = pd.DataFrame(
    size_rows
)

display(
    size_recall_df
)


In [ ]:
# CELL 22 — Robustness test: blur, low light, lower resolution

ROBUST_ROOT = Path(
    "/kaggle/working/robustness_tests"
)

ROBUST_SAMPLE_SIZE = min(
    150,
    len(test_images)
)

if ROBUST_ROOT.exists():
    shutil.rmtree(
        ROBUST_ROOT
    )

rng = random.Random(
    SEED
)

robust_sample = rng.sample(
    test_images,
    ROBUST_SAMPLE_SIZE
)

def make_variant_image(
    image,
    variant
):

    if variant == "blur":
        return cv2.GaussianBlur(
            image,
            (7, 7),
            0
        )

    if variant == "dark":
        return cv2.convertScaleAbs(
            image,
            alpha=0.65,
            beta=-15
        )

    if variant == "low_resolution":

        h, w = image.shape[:2]

        small = cv2.resize(
            image,
            (
                max(32, w // 3),
                max(32, h // 3)
            ),
            interpolation=cv2.INTER_AREA
        )

        return cv2.resize(
            small,
            (w, h),
            interpolation=cv2.INTER_LINEAR
        )

    raise ValueError(
        variant
    )

robust_yamls = {}

for variant in [
    "blur",
    "dark",
    "low_resolution"
]:

    root = (
        ROBUST_ROOT /
        variant
    )

    img_dir = (
        root /
        "images"
    )

    lbl_dir = (
        root /
        "labels"
    )

    img_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    lbl_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for img_path in robust_sample:

        image = cv2.imread(
            str(img_path)
        )

        degraded = make_variant_image(
            image,
            variant
        )

        out_img = (
            img_dir /
            img_path.name
        )

        out_lbl = (
            lbl_dir /
            img_path.with_suffix(".txt").name
        )

        cv2.imwrite(
            str(out_img),
            degraded
        )

        source_label = (
            test_lbl_dir /
            img_path.with_suffix(".txt").name
        )

        shutil.copy2(
            source_label,
            out_lbl
        )

    variant_yaml = (
        root /
        "data.yaml"
    )

    with open(
        variant_yaml,
        "w"
    ) as f:

        yaml.safe_dump(
            {
                "path": str(root),
                "val": "images",
                "names": main_ds["names"],
            },
            f,
            sort_keys=False,
        )

    robust_yamls[
        variant
    ] = variant_yaml

robust_rows = []

for variant, variant_yaml in robust_yamls.items():

    metrics = best_model.val(
        data=str(variant_yaml),
        split="val",

        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,

        conf=SELECTED_CONF,

        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),

        plots=False,
        verbose=False,
    )

    robust_rows.append({
        "condition": variant,
        "precision": float(
            metrics.box.mp
        ),
        "recall": float(
            metrics.box.mr
        ),
        "mAP50": float(
            metrics.box.map50
        ),
        "mAP50-95": float(
            metrics.box.map
        ),
    })

robustness_df = pd.DataFrame(
    robust_rows
)

display(
    robustness_df
)


## CELL 23 — External generalization

If a second YOLO dataset is attached, this test uses **exactly matching classes only**.

Example:
- `tire` ↔ `tyre` is treated as the same spelling.
- `water tank` ↔ `water tank` is valid.
- `drum` is **not automatically treated as bucket**.
- `open container` is **not automatically treated as bottle**.

That keeps the external test scientifically cleaner.


In [ ]:
# CELL 23 — External generalization on exact common classes

def canonical_class_name(name):

    name = (
        str(name)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    name = " ".join(
        name.split()
    )

    if name == "tyre":
        name = "tire"

    return name

external_result_df = None

if EXTERNAL_YAML is None:

    print(
        "No external dataset attached. "
        "Skipping external generalization test."
    )

else:

    ext_ds = load_dataset_yaml(
        EXTERNAL_YAML
    )

    main_name_to_id = {
        canonical_class_name(name): cid
        for cid, name
        in main_ds["names"].items()
    }

    ext_name_to_id = {
        canonical_class_name(name): cid
        for cid, name
        in ext_ds["names"].items()
    }

    common_names = sorted(
        set(main_name_to_id) &
        set(ext_name_to_id)
    )

    print(
        "Exact common classes:",
        common_names
    )

    if not common_names:

        print(
            "No exact common classes. "
            "External quantitative validation skipped."
        )

    else:

        EXT_ROOT = Path(
            "/kaggle/working/external_common"
        )

        if EXT_ROOT.exists():
            shutil.rmtree(
                EXT_ROOT
            )

        ext_img_out = (
            EXT_ROOT /
            "images"
        )

        ext_lbl_out = (
            EXT_ROOT /
            "labels"
        )

        ext_img_out.mkdir(
            parents=True,
            exist_ok=True
        )

        ext_lbl_out.mkdir(
            parents=True,
            exist_ok=True
        )

        # Prefer external test; otherwise use validation.
        if (
            ext_ds["test_images"] is not None
            and
            Path(ext_ds["test_images"]).exists()
        ):
            ext_source_images = (
                ext_ds["test_images"]
            )

        else:
            ext_source_images = (
                ext_ds["val_images"]
            )

        ext_source_labels = (
            image_dir_to_label_dir(
                ext_source_images
            )
        )

        main_common_ids = []
        ext_to_main = {}

        for name in common_names:

            main_id = (
                main_name_to_id[name]
            )

            ext_id = (
                ext_name_to_id[name]
            )

            ext_to_main[
                ext_id
            ] = main_id

            main_common_ids.append(
                main_id
            )

        kept_images = 0
        kept_boxes = 0

        for img_path in list_images(
            ext_source_images
        ):

            rel = img_path.relative_to(
                ext_source_images
            )

            src_label = (
                ext_source_labels /
                rel
            ).with_suffix(
                ".txt"
            )

            if not src_label.exists():
                continue

            mapped_lines = []

            for (
                cid,
                x,
                y,
                w,
                h
            ) in read_yolo_labels(
                src_label
            ):

                if cid in ext_to_main:

                    new_cid = (
                        ext_to_main[cid]
                    )

                    mapped_lines.append(
                        f"{new_cid} "
                        f"{x:.6f} "
                        f"{y:.6f} "
                        f"{w:.6f} "
                        f"{h:.6f}"
                    )

                    kept_boxes += 1

            flat_name = "__".join(
                rel.parts
            )

            dst_img = (
                ext_img_out /
                flat_name
            )

            dst_lbl = (
                ext_lbl_out /
                Path(flat_name)
                .with_suffix(".txt")
            )

            os.symlink(
                img_path,
                dst_img
            )

            dst_lbl.write_text(
                "\n".join(mapped_lines)
                +
                ("\n" if mapped_lines else "")
            )

            kept_images += 1

        EXT_COMMON_YAML = (
            EXT_ROOT /
            "data.yaml"
        )

        with open(
            EXT_COMMON_YAML,
            "w"
        ) as f:

            yaml.safe_dump(
                {
                    "path": str(EXT_ROOT),
                    "val": "images",
                    "names": main_ds["names"],
                },
                f,
                sort_keys=False,
            )

        print(
            "External images prepared:",
            kept_images
        )

        print(
            "Common-class boxes kept:",
            kept_boxes
        )

        print(
            "Main-model class IDs evaluated:",
            main_common_ids
        )

        ext_metrics = best_model.val(
            data=str(EXT_COMMON_YAML),
            split="val",

            classes=main_common_ids,

            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,

            conf=SELECTED_CONF,

            device=(
                0
                if torch.cuda.is_available()
                else "cpu"
            ),

            plots=True,
            verbose=False,

            project=str(RUNS_DIR),
            name="external_generalization",
            exist_ok=True,
        )

        external_result_df = pd.DataFrame([
            {
                "common_classes":
                    ", ".join(common_names),

                "precision":
                    float(ext_metrics.box.mp),

                "recall":
                    float(ext_metrics.box.mr),

                "mAP50":
                    float(ext_metrics.box.map50),

                "mAP50-95":
                    float(ext_metrics.box.map),
            }
        ])

        display(
            external_result_df
        )


In [ ]:
# CELL 24 — Save and display sample test predictions

SAMPLE_PRED_DIR = Path(
    "/kaggle/working/sample_predictions"
)

if SAMPLE_PRED_DIR.exists():
    shutil.rmtree(
        SAMPLE_PRED_DIR
    )

sample_count = min(
    12,
    len(test_images)
)

sample_test_images = (
    random.Random(SEED)
    .sample(
        test_images,
        sample_count
    )
)

prediction_results = best_model.predict(
    source=[
        str(p)
        for p in sample_test_images
    ],

    conf=SELECTED_CONF,
    iou=0.60,
    imgsz=IMAGE_SIZE,

    save=True,

    project=str(
        SAMPLE_PRED_DIR
    ),

    name="predictions",
    exist_ok=True,

    verbose=False,

    device=(
        0
        if torch.cuda.is_available()
        else "cpu"
    ),
)

cols = 3
rows = math.ceil(
    sample_count / cols
)

plt.figure(
    figsize=(16, 5 * rows)
)

for i, result in enumerate(
    prediction_results,
    start=1
):

    annotated = result.plot()

    annotated_rgb = cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )

    plt.subplot(
        rows,
        cols,
        i
    )

    plt.imshow(
        annotated_rgb
    )

    plt.axis(
        "off"
    )

    plt.title(
        Path(result.path).name
    )

plt.tight_layout()
plt.show()

print(
    "Saved predictions:",
    SAMPLE_PRED_DIR
)


In [ ]:
# CELL 25 — Optional Dhaka local validation

dhaka_images = []

for p in INPUT_ROOT.rglob("*"):

    if (
        p.is_file()
        and
        p.suffix.lower() in IMAGE_EXTS
        and
        "dhaka" in str(p.parent).lower()
    ):
        dhaka_images.append(p)

dhaka_images = sorted(
    dhaka_images
)

if not dhaka_images:

    print(
        "No Dhaka image folder found. "
        "Skipping local validation."
    )

else:

    print(
        "Dhaka images found:",
        len(dhaka_images)
    )

    DHAKA_OUT = Path(
        "/kaggle/working/dhaka_predictions"
    )

    best_model.predict(
        source=[
            str(p)
            for p in dhaka_images
        ],

        conf=SELECTED_CONF,
        iou=0.60,
        imgsz=IMAGE_SIZE,

        save=True,

        project=str(
            DHAKA_OUT
        ),

        name="predictions",
        exist_ok=True,

        verbose=False,

        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),
    )

    print(
        "Dhaka predictions saved:",
        DHAKA_OUT
    )


In [ ]:
# CELL 26 — Speed and resource test

speed_sample = (
    test_images[
        :min(
            100,
            len(test_images)
        )
    ]
)

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

_ = best_model.predict(
    source=[
        str(p)
        for p in speed_sample
    ],

    conf=SELECTED_CONF,
    iou=0.60,
    imgsz=IMAGE_SIZE,

    batch=BATCH_SIZE,

    verbose=False,

    device=(
        0
        if torch.cuda.is_available()
        else "cpu"
    ),
)

elapsed = (
    time.perf_counter() -
    start
)

n = len(
    speed_sample
)

seconds_per_image = (
    elapsed / n
)

images_per_minute = (
    60 / seconds_per_image
)

model_size_mb = (
    BEST_PT.stat().st_size /
    (1024 * 1024)
)

print(
    f"Images tested:       {n}"
)

print(
    f"Total time:          {elapsed:.2f} s"
)

print(
    f"Seconds per image:   {seconds_per_image:.3f}"
)

print(
    f"Images per minute:   {images_per_minute:.1f}"
)

print(
    f"Model size:          {model_size_mb:.1f} MB"
)

if torch.cuda.is_available():

    peak_gpu_mb = (
        torch.cuda.max_memory_allocated() /
        (1024 * 1024)
    )

    print(
        f"Peak GPU memory:     {peak_gpu_mb:.1f} MB"
    )


In [ ]:
# CELL 27 — Export final model and reports

EXPORT_DIR = Path(
    "/kaggle/working/mosquitowatch_export"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MODEL = (
    EXPORT_DIR /
    "mosquitowatch_yolo26s_best.pt"
)

shutil.copy2(
    BEST_PT,
    FINAL_MODEL
)

quality_df.to_csv(
    EXPORT_DIR /
    "data_quality_summary.csv",
    index=False
)

augmentation_df.to_csv(
    EXPORT_DIR /
    "augmentation_before_after.csv",
    index=False
)

fit_metrics_df.to_csv(
    EXPORT_DIR /
    "train_val_test_metrics.csv",
    index=False
)

per_class_df.to_csv(
    EXPORT_DIR /
    "per_class_test_metrics.csv",
    index=False
)

threshold_df.to_csv(
    EXPORT_DIR /
    "confidence_threshold_tuning.csv",
    index=False
)

size_recall_df.to_csv(
    EXPORT_DIR /
    "object_size_recall.csv",
    index=False
)

robustness_df.to_csv(
    EXPORT_DIR /
    "robustness_metrics.csv",
    index=False
)

if external_result_df is not None:

    external_result_df.to_csv(
        EXPORT_DIR /
        "external_generalization_metrics.csv",
        index=False
    )

experiment_summary = {
    "model": "YOLO26s",
    "image_size": IMAGE_SIZE,

    "selected_confidence_threshold":
        SELECTED_CONF,

    "train_images_before_augmentation":
        len(train_image_info),

    "train_images_after_augmentation":
        len(list_images(train_img_dir)),

    "best_model_path":
        str(FINAL_MODEL),
}

with open(
    EXPORT_DIR /
    "experiment_summary.json",
    "w"
) as f:

    json.dump(
        experiment_summary,
        f,
        indent=2
    )

ZIP_PATH = shutil.make_archive(
    "/kaggle/working/mosquitowatch_export",
    "zip",
    EXPORT_DIR
)

print(
    "Final model:",
    FINAL_MODEL
)

print(
    "Export folder:",
    EXPORT_DIR
)

print(
    "ZIP file:",
    ZIP_PATH
)

print(
    "\nDownload the ZIP and best.pt "
    "from Kaggle's Output section."
)


# What you should record for your hackathon

### Data quality
- image count
- corrupted/invalid files
- class distribution
- small/medium/large object distribution
- duplicate/leakage findings

### Augmentation
- training images before
- training images after
- class box counts before
- class box counts after

### Model
- Precision
- Recall
- mAP@50
- mAP@50–95
- per-class metrics

### Fit
- underfitting check
- overfitting check
- train/validation/test gap

### Drone reliability
- small-object recall
- blur robustness
- low-light robustness
- lower-resolution robustness
- external dataset generalization
- optional Dhaka validation

### Deployment
- selected confidence threshold
- seconds per image
- images per minute
- model size

### Final Layer 1 output
`mosquitowatch_yolo26s_best.pt`

Layer 2 will later receive:
- annotated drone image
- object counts
